# 🚀 ComfyUI Colab Ultra (Bản Nâng Cấp & Tối Ưu Toàn Diện)
### Hỗ Trợ Đầy Đủ Mô Hình AI Mới Nhất, Background Removal, Video Gen, Đồng Bộ 2 Chiều Pixaroma (Alt + W), Smart Live Backup (.tar.zst) & SSH-Tailscale Local Tunnel
*Tối ưu hóa chuyên sâu cho Google Colab: giải phóng RAM kernel, nén sao lưu Zstandard đa luồng, giữ 100% Models/Input/Output/Workflows trên Google Drive và hỗ trợ kết nối SSH-Tailscale thẳng về `http://127.0.0.1:8188` trên máy cá nhân.*

___

#### 💡 Hướng dẫn nhanh:
1. Chạy **Bước 1** để kết nối Google Drive, cài đặt ComfyUI & Custom Nodes chính thống mới nhất, kích hoạt Động cơ Đồng bộ 1 Chiều Output -> Input và Smart Live Backup.
2. Chạy **Bước 2** để khởi động ComfyUI qua **Colab-Proxy** (mở link trực tiếp), **SSH-Tailscale** (chạy thẳng về `http://127.0.0.1:8188` trên máy cá nhân + VS Code Remote-SSH), hoặc **Cloudflare**.


In [ ]:
#@title 🛠️ 1. Cài đặt môi trường & Đồng bộ Google Drive

Connect_Drive = True #@param {type:"boolean"}
ComfyUI_Data_Path = "/content/drive/MyDrive/ComfyUI_Data" #@param {type:"string"}
Use_Drive_Backup = True #@param {type:"boolean", label:"Kích hoạt Smart Live Backup cho Custom Nodes"}
Force_Reinstall = False #@param {type:"boolean"}
Update_Nodes = True #@param {type:"boolean", label:"Tự động cập nhật ComfyUI & Custom Nodes lên bản mới nhất"}
Install_Video_Nodes = True #@param {type:"boolean"}

import os
import sys
import subprocess

REPO_URL = "https://github.com/nguyenducvuongg/ComfyUIonColab.git"
REPO_DIR = "/content/ComfyUIonColab"

# Tự động đồng bộ mã nguồn ComfyColab mới nhất từ GitHub
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=False)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "HEAD", "-q"], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD", "-q"], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Xóa cache module comfycolab để luôn nạp mã nguồn mới nhất
for mod in list(sys.modules.keys()):
    if mod == 'comfycolab' or mod.startswith('comfycolab.'):
        del sys.modules[mod]

from comfycolab import launch_setup

launch_setup(
    connect_drive=Connect_Drive,
    drive_path=ComfyUI_Data_Path,
    use_backup=Use_Drive_Backup,
    update_nodes=Update_Nodes,
    install_video_nodes=Install_Video_Nodes,
    force_reinstall=Force_Reinstall
)


In [ ]:
#@title 🚀 2. Khởi chạy ComfyUI & Đường truyền (Colab-Proxy / SSH-Tailscale / Cloudflare)

Tunnel_Method = "Colab-Proxy" #@param ["Colab-Proxy", "SSH-Tailscale", "Cloudflare"]
Tailscale_Auth_Key = "" #@param {type:"string"}
Tailscale_Hostname = "colab-comfy" #@param {type:"string"}
SSH_Public_Key = "" #@param {type:"string"}
Local_Only_Bind = True #@param {type:"boolean", label:"Chỉ bind 127.0.0.1 khi chạy SSH-Tailscale (khuyên dùng để bảo mật)"}
Extra_Args = "--preview-method auto --enable-cors-header" #@param {type:"string"}
Port = 8188 #@param {type:"integer"}

# Tự động làm mới module comfycolab trước khi khởi chạy
import os
import sys
import subprocess

REPO_URL = "https://github.com/nguyenducvuongg/ComfyUIonColab.git"
REPO_DIR = "/content/ComfyUIonColab"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=False)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "HEAD", "-q"], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD", "-q"], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for mod in list(sys.modules.keys()):
    if mod == 'comfycolab' or mod.startswith('comfycolab.'):
        del sys.modules[mod]

from comfycolab import start_comfyui_and_tunnel

start_comfyui_and_tunnel(
    method=Tunnel_Method,
    port=Port,
    tailscale_auth_key=Tailscale_Auth_Key,
    tailscale_hostname=Tailscale_Hostname,
    ssh_public_key=SSH_Public_Key,
    local_only_bind=Local_Only_Bind,
    extra_args=Extra_Args
)
